# Calculating yields

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you can:

1. Calculate the theoretical maximum yield of a product from an FBA solution.
2. Convert between molar, C-mol and mass yields.
3. Calculate the biomass yield on a substrate.
4. Quantify the trade-off between growth and product yield, and how it changes without oxygen.

**Estimated time:** 70 min (15 min guided, 55 min exercises).

**Prerequisites:** notebooks 01, 02 and 05 (FBA, objectives, bounds).

### Background

The **yield** $Y_{P/S}$ of a product $P$ on a substrate $S$ is how much product is made per amount of substrate consumed. In a flux model both are fluxes (mmol gDW⁻¹ h⁻¹), so the yield is simply their ratio:

$$
Y_{P/S} = \frac{v_P}{-v_S}
$$

(the minus sign because substrate uptake is a negative exchange flux). The same yield can be expressed in different units ([Villadsen et al., 2011](https://doi.org/10.1007/978-1-4419-9688-6)):

* **molar yield**: mol product / mol substrate;
* **C-mol yield**: mol carbon in product / mol carbon in substrate. This tells you which fraction of the substrate carbon ends up in the product; it cannot exceed 1;
* **mass yield**: g product / g substrate, obtained with the molecular weights. This is what matters economically.

The **theoretical maximum yield** is the yield when the cell does nothing but make the product (growth set to zero) ([Orth et al., 2010](https://doi.org/10.1038/nbt.1614)). Real processes need growth and maintenance, so the achievable yield is lower. How much lower is shown by the trade-off between growth rate and product yield, the same information as in a phenotypic phase plane (notebook 04).

In [ ]:
from cobra.io import read_sbml_model
model_original = read_sbml_model('data/iJO1366.xml.gz')

## Product yield

We use acetate as the example product. We work on a copy of the model so the original stays untouched.

In [ ]:
model = model_original.copy()

First, make the product exchange reaction the objective of the model, so FBA maximizes acetate secretion instead of growth.

In [ ]:
model.objective = model.reactions.EX_ac_e

Then determine the maximum production flux by solving the model.

In [ ]:
solution = model.optimize()
solution.fluxes['EX_ac_e']

Dividing the production flux by the uptake flux of the carbon source (glucose) gives the molar yield (mol acetate / mol glucose). Glucose uptake is negative, hence the `-1.`. Since the growth rate is zero in this optimum, this is the theoretical maximum yield.

In [ ]:
solution.fluxes['EX_ac_e'] / (-1. * solution.fluxes['EX_glc__D_e'])

## Exercise (15 min)

* Convert the yield into a C-mol (product) / C-mol (carbon source) yield.
* Calculate a mass yield instead of molar yield (g (product) / g (carbon source))

In [ ]:
# Your code here


## Biomass yield

The biomass reaction is scaled such that its flux is the growth rate in h⁻¹, i.e. gDW of new biomass per gDW per hour. Dividing it by the glucose uptake flux therefore gives the biomass yield in gDW per mmol glucose.

In [ ]:
model = model_original.copy()

In [ ]:
solution = model.optimize()
g_dw_biomass_per_mmol_glucose = solution.objective_value / (-1. * solution.fluxes['EX_glc__D_e'])
g_dw_biomass_per_mmol_glucose

## Exercise (10 min)

The units of this yield are gDW (Biomass) / mmol (glucose). Convert this number into a mass yield.

In [ ]:
# Your code here


## Growth vs. product yield

## Exercise (30 min)

In the previous section we calculated theoretical maximum yields in the absence of competing objectives. But how does product yield vary with growth?

* Determine molar, mass, and C-mol yields of acetate for the following growth rates (plot them using the code snippets below).
* How do the yields look under anaerobic conditions?

In [ ]:
model = model_original.copy()

In [ ]:
growth_rates = [0.0, 0.10915242, 0.21830485, 0.32745727, 0.43660969, 0.54576212, 0.65491454, 0.76406697, 0.87321939, 0.98237181]

In [ ]:
# Replace these placeholders with your own values (one per growth rate)
product_fluxes = [0] * len(growth_rates)
yields = [0] * len(growth_rates)

In [ ]:
# Your code here


Plot the product fluxes against the growth rate:

In [ ]:
import matplotlib.pyplot as plt
plt.plot(growth_rates, product_fluxes)
plt.xlabel('Growth [$h^{-1}$]')
plt.ylabel('Product flux [$mmol gDW^{-1} h^{-1}$]')
plt.show()

Plot the product yields against the growth rate:

In [ ]:
# Your code here


In [ ]:
plt.plot(growth_rates, yields)
plt.xlabel('Growth [$h^{-1}$]')
plt.ylabel('Product yield [$mmol(acetate) / mmol(glucose)$]')
plt.show()

In [ ]:
# Your code here


## References

- Orth, Thiele, Palsson (2010). What is flux balance analysis? *Nature Biotechnology* 28:245–248. [doi.org/10.1038/nbt.1614](https://doi.org/10.1038/nbt.1614)
- Villadsen, Nielsen, Lidén (2011). Bioreaction Engineering Principles. *Springer US*. [doi.org/10.1007/978-1-4419-9688-6](https://doi.org/10.1007/978-1-4419-9688-6)